In [1]:
import pandas as pd
BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"
df = pd.read_csv(BASE_PATH + "cell2024_final.csv")
df = df[df["mutant_sequence"].notna()].copy()
df_eval = df[df["Mislocalized"].notna()].copy()
df_pos = df_eval[df_eval["Mislocalized"] == 1]
phenotypes = df_pos["Mislocalization phenotype"].value_counts()
print(phenotypes.to_string())

Mislocalization phenotype
Plasma membrane>ER                       54
Golgi apparatus>ER                       27
Cytoplasm>Foci                           10
Cytoplasm>Nucleus                        10
Intermediate filaments>Foci               9
Plasma membrane>Golgi apparatus           9
ER>Foci                                   8
ER>ER                                     8
Cytoplasm, nucleus>Cytoplasm              7
Vesicles>ER                               7
Cellular periphery>Cytoplasm              6
ER>Plasma membrane                        6
Foci>Cytoplasm                            4
Nucleus>Cytoplasm                         4
Mitochondria>Cytoplasm, nucleus           4
Tubulin>Cytoplasm                         4
Foci>ER                                   4
Cytoplasm, nucleus>Foci                   3
Mitochondria>Cytoplasm                    3
Plasma membrane>Cytoplasm                 3
Mitochondria>Mitochondria                 3
Nucleus>Foci                              3
ER>Gol

In [2]:
def assign_class(phenotype):
    if not isinstance(phenotype, str):
        return None
    parts = phenotype.split(">")
    if len(parts) != 2:
        return None
    wt = parts[0].strip()
    mt = parts[1].strip()

    # C1：no relocation
    if wt == mt:
        return "C1_no_reloc"
    # C2：aggregation
    if mt in {"Foci", "Rods & rings"}:
        return "C2_aggregation"
    # C3：secretory pathway relocation
    if mt in {"ER", "Golgi apparatus", "Vesicles", "Plasma membrane",
              "Nuclear membrane", "Nuclear periphery"}:
        return "C3_secretory"
    # C4：nuclear relocation
    if mt in {"Nucleus", "Nucleolus", "Cytoplasm, nucleus"}:
        return "C4_nuclear"
    # C5： cytoplasmic relocation / others
    if mt in {"Cytoplasm", "Cellular periphery", "Mitochondria"}:
        return "C5_cytoplasmic"

    return "C_unknown"

df["label_5class"] = df["Mislocalization phenotype"].apply(assign_class)

# Curated assignment based on the Additional benign variants localisation columns:
# WT RPE65 localises to cytoplasm + foci, whereas K294T localises to cytoplasm only.
rpe65_k294t = (
    df["Gene"].eq("RPE65")
    & df["Mutation_used"].eq("K294T")
    & df["source"].eq("additional_benign")
    & df["Mislocalized"].eq(1)
)
assert rpe65_k294t.sum() == 1, (
    f"Expected exactly one positive additional_benign RPE65 K294T row; "
    f"found {rpe65_k294t.sum()}"
)
df.loc[rpe65_k294t, "label_5class"] = "C5_cytoplasmic"

# 验证
df_pos = df[df["Mislocalized"] == 1]
print("=== Five Classes ===")
print(df_pos["label_5class"].value_counts())

print(f"\nNot covered (C_unknown): {(df_pos['label_5class'] == 'C_unknown').sum()}")
print(f"label is None: {df_pos['label_5class'].isna().sum()}")

df_multiclass = df[
    (df["Mislocalized"] == 0) | 
    (df["label_5class"].notna())
].copy()

print(f"Available rows: {len(df_multiclass)}")
print(f"Positive: {(df_multiclass['Mislocalized']==1).sum()}")
print(f"Negative: {(df_multiclass['Mislocalized']==0).sum()}")

df.to_csv(BASE_PATH + "cell2024_final_with_labels.csv", index=False)
print("\nSaved to cell2024_final_with_labels.csv")

=== Five Classes ===
label_5class
C3_secretory      121
C5_cytoplasmic     39
C2_aggregation     34
C4_nuclear         29
C1_no_reloc        13
Name: count, dtype: int64

Not covered (C_unknown): 0
label is None: 0
Available rows: 2179
Positive: 236
Negative: 1943

Saved to cell2024_final_with_labels.csv
